In [1]:
# import dependencies
import pandas as pd
import re
import numpy as np
from rapidfuzz import process

In [2]:
# load acs data
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")

Used this [list of HMGP activities](https://www.fema.gov/sites/default/files/2020-08/fema_mt-egrants-guide-to-eligible-activities-and-codes_job_aid_March_2018.pdf) to create a csv to classify projectType.

In [3]:
# load FEMA project codes
codes_df = pd.read_csv("../data/resources/FEMA_project_codes.csv")
codes_df.head()

,code,main_category,subcategory,flood_mitigation_flag
0,100.1,planning,public_awareness,0
1,103.1,planning,feasibility_study,0
2,104.1,planning,codes_standards,0
3,106.1,planning,other_nonconstruction,0
4,106.2,planning,other_nonconstruction,0


In [4]:
# load FEMA Hazard Mitigation Assistance Projects for Vermont
df = pd.read_csv("../data/raw/funding/vt_hma_awards.csv")
df

,projectIdentifier,programArea,programFy,region,state,stateNumberCode,county,countyCode,disasterNumber,projectCounties,...,federalShareObligated,subrecipientAdminCostAmt,srmcObligatedAmt,recipientAdminCostAmt,costSharePercentage,benefitCostRatio,netValueBenefits,numberOfFinalProperties,numberOfProperties,id
0,DR-4532-0005-R,HMGP,2020,1,Vermont,50,Rutland,21.0,4532.0,RUTLAND,...,140188.70,0.0,0.0,0.0,0.90,1.13,323000.0,1,1,344986e8-9849-4f9a-9504-436d4a3c92c7
1,DR-4720-0047-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,237540.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,1bce9785-0760-4558-bf79-6cd8047bad9b
2,DR-4720-0069-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,265767.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,62892290-55db-4565-90e1-29436e27138c
3,DR-4720-0043-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,121983.00,0.0,0.0,0.0,0.75,0.00,0.0,1,1,65ee2247-4a87-42b0-a1d0-70c290c0a844
4,DR-4022-0109-R,HMGP,2011,1,Vermont,50,Washington,23.0,4022.0,WASHINGTON,...,0.00,0.0,0.0,0.0,0.75,0.00,0.0,0,1,d3018dab-e76a-4236-8d76-a5da6fc581ff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,DR-1428-0014-R,HMGP,2002,1,Vermont,50,Essex,9.0,1428.0,ESSEX,...,6525.00,261.0,0.0,135.0,0.75,6.39,55555.0,0,0,9cf2e7d5-7acb-4bae-b1da-e54aa56e0a3f
752,DR-4022-0153-R,HMGP,2011,1,Vermont,50,Franklin,11.0,4022.0,FRANKLIN,...,71110.52,0.0,0.0,0.0,0.75,1.13,130933.0,0,0,6be65b91-9c1f-420e-9127-1a242b92b0c7
753,DR-1559-0003-R,HMGP,2004,1,Vermont,50,Franklin,11.0,1559.0,FRANKLIN,...,18970.00,435.0,0.0,47.0,0.75,1.62,90925.0,0,0,6c5d37d8-0808-4882-ab16-e42063fbcd29
754,DR-1698-0001-R,HMGP,2007,1,Vermont,50,Franklin,11.0,1698.0,FRANKLIN,...,56364.00,0.0,0.0,0.0,0.75,1.15,123837.0,0,0,3209cad2-8c94-443d-94ba-2bd02b2b377c


In [5]:
# check for negative values in federalShareObligated
# df[df["federalShareObligated"] < 0]

In [6]:
# check distribution of federalShareObligated
# highly skewed, with a long tail of large values
#  median is much lower than mean
df["federalShareObligated"].describe()

count    6.550000e+02
mean     1.662356e+05
std      3.753405e+05
min      0.000000e+00
25%      9.479375e+03
50%      4.326200e+04
75%      1.706199e+05
max      4.220919e+06
Name: federalShareObligated, dtype: float64

In [7]:
# merge project codes csv with FEMA HMA dataframe, including handling multiple codes in projectType

# extract all codes from projectType, creating a list of codes for each project
# regex: one or more digits, followed by a period, followed by one or more digits (e.g. 200.1)
df["codes"] = df["projectType"].str.findall(r"\d+\.\d+")

# explode codes so each project-code is a row
df_exploded = df.explode("codes")

# ensure string type for merging
df_exploded["codes"] = df_exploded["codes"].astype(str)
codes_df["code"] = codes_df["code"].astype(str)

# merge exploded df with project type codebook
df_merged = df_exploded.merge(codes_df, left_on="codes", right_on="code", how="left")

# aggregate back to project level, combining main_category and subcategory for projects with multiple codes
# set flood_mitigation_flag to 1 if any code for the project has a flag of 1, otherwise 0
agg = (
    df_merged.groupby("projectIdentifier")
    .agg(
        {
            "flood_mitigation_flag": "max",
            "main_category": lambda x: ",".join(sorted(set(x.dropna()))),
            "subcategory": lambda x: ",".join(sorted(set(x.dropna()))),
        }
    )
    .reset_index()
)

# merge back to original df
df_coded = df.merge(agg, on="projectIdentifier", how="left")
df = df_coded
df

,projectIdentifier,programArea,programFy,region,state,stateNumberCode,county,countyCode,disasterNumber,projectCounties,...,costSharePercentage,benefitCostRatio,netValueBenefits,numberOfFinalProperties,numberOfProperties,id,codes,flood_mitigation_flag,main_category,subcategory
0,DR-4532-0005-R,HMGP,2020,1,Vermont,50,Rutland,21.0,4532.0,RUTLAND,...,0.90,1.13,323000.0,1,1,344986e8-9849-4f9a-9504-436d4a3c92c7,[200.1],1.0,acquisition_buyouts,acquisition
1,DR-4720-0047-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,0.75,0.00,0.0,1,1,1bce9785-0760-4558-bf79-6cd8047bad9b,[200.1],1.0,acquisition_buyouts,acquisition
2,DR-4720-0069-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,0.75,0.00,0.0,1,1,62892290-55db-4565-90e1-29436e27138c,[200.1],1.0,acquisition_buyouts,acquisition
3,DR-4720-0043-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,0.75,0.00,0.0,1,1,65ee2247-4a87-42b0-a1d0-70c290c0a844,[200.1],1.0,acquisition_buyouts,acquisition
4,DR-4022-0109-R,HMGP,2011,1,Vermont,50,Washington,23.0,4022.0,WASHINGTON,...,0.75,0.00,0.0,0,1,d3018dab-e76a-4236-8d76-a5da6fc581ff,[202.1],1.0,structure_protection,elevation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,DR-1428-0014-R,HMGP,2002,1,Vermont,50,Essex,9.0,1428.0,ESSEX,...,0.75,6.39,55555.0,0,0,9cf2e7d5-7acb-4bae-b1da-e54aa56e0a3f,[403.1],1.0,infrastructure_stormwater,stormwater_culverts
752,DR-4022-0153-R,HMGP,2011,1,Vermont,50,Franklin,11.0,4022.0,FRANKLIN,...,0.75,1.13,130933.0,0,0,6be65b91-9c1f-420e-9127-1a242b92b0c7,[403.1],1.0,infrastructure_stormwater,stormwater_culverts
753,DR-1559-0003-R,HMGP,2004,1,Vermont,50,Franklin,11.0,1559.0,FRANKLIN,...,0.75,1.62,90925.0,0,0,6c5d37d8-0808-4882-ab16-e42063fbcd29,[403.1],1.0,infrastructure_stormwater,stormwater_culverts
754,DR-1698-0001-R,HMGP,2007,1,Vermont,50,Franklin,11.0,1698.0,FRANKLIN,...,0.75,1.15,123837.0,0,0,3209cad2-8c94-443d-94ba-2bd02b2b377c,[403.1],1.0,infrastructure_stormwater,stormwater_culverts


In [8]:
# check rows where projectType contains more than one code
# multi_code_mask = df['codes'].apply(lambda x: isinstance(x, list) and len(x) > 1)
# df[multi_code_mask][['projectIdentifier', 'projectType', 'codes', 'main_category', 'subcategory']]

In [9]:
# check NaN codes, projectType
# nan_code_mask = df['codes'].apply(lambda x: (isinstance(x, list) and len(x) == 0) or (isinstance(x, float) and np.isnan(x))) | df['projectType'].isna()
# df[nan_code_mask][['projectIdentifier', 'projectType', 'codes', 'main_category', 'subcategory']]

In [10]:
# df['main_category'].value_counts()

In [11]:
# df['flood_mitigation_flag'].value_counts()

In [12]:
# check for multiple counties in projectCounties
# df[df["projectCounties"].str.contains(" ", na=False)].drop(['projectIdentifier', 'programArea', 'programFy', 'region', 'state',
#        'stateNumberCode'], axis=1)

In [13]:
# Barre, Rutland, St. Albans, and Newport have a city and town of the same name
# Vermont is sometimes lacking in creativity
# df[
#     df["subrecipient"].str.contains("BARRE|RUTLAND|ALBANS|NEWPORT", case=False, na=False)
# ]["subrecipient"].value_counts()

In [14]:
# df[df["subrecipient"].str.contains("BARRE|RUTLAND|ALBANS|NEWPORT", case=False, na=False)][['subrecipient', 'projectCounties', 'projectType', 'codes', 'flood_mitigation_flag', 'main_category', 'subcategory']]

In [15]:
# dict for manual town assignments based on subrecipient names
manual_town_dict = {
    "JEFFERSONVILLE": "CAMBRIDGE",
    "LYNDONVILLE": "LYNDON",
    "ENOSBURG FALLS": "ENOSBURG",
    "RUPERT- HIGHWAY DEPARTMENT": "RUPERT",
    "STOWE ELECTRIC DEPARTMENT": "STOWE",
    "MONTPELIER- PUBLIC WORKS DEPARTMENT": "MONTPELIER CITY",
    "BRATTLEBORO HOUSING AUTHORITY": "BRATTLEBORO",
    "BARRE HISTORICAL SOCIETY INC": "BARRE",
}


# Barre, Rutland, St. Albans, and Newport have a city and town of the same name
# Vermont is sometimes lacking in creativity
# override function for classifying a few towns with ambiguous names, based on explicit mentions of "TOWN" or "CITY" in the subrecipient name
def classify_barre_rutland_stalbans(subrecipient):
    # Explicit city/town assignment
    if pd.isna(subrecipient):
        return None
    s = subrecipient.upper()
    if "BARRE" in s:
        if "TOWN" in s:
            return "BARRE TOWN"
        elif "CITY" in s:
            return "BARRE CITY"
        elif s.strip() == "BARRE":
            return "AMBIGUOUS: BARRE"
    if "RUTLAND" in s:
        if "TOWN" in s:
            return "RUTLAND TOWN"
        elif "CITY" in s:
            return "RUTLAND CITY"
        elif s.strip() == "RUTLAND":
            return "AMBIGUOUS: RUTLAND"
    if "ST. ALBANS" in s or "ST ALBANS" in s:
        if "TOWN" in s:
            return "ST. ALBANS TOWN"
        elif "CITY" in s:
            return "ST. ALBANS CITY"
        elif s.replace(".", "").strip() in ["ST ALBANS", "ST ALBANS"]:
            return "AMBIGUOUS: ST. ALBANS"
    if "NEWPORT" in s:
        if "TOWN" in s:
            return "NEWPORT TOWN"
        elif "CITY" in s:
            return "NEWPORT CITY"
        elif s.strip() == "NEWPORT":
            return "AMBIGUOUS: NEWPORT"
    return None

In [16]:
# regex to clean subrecipient names
def clean_town(x):
    # handle NaN values
    if pd.isna(x):
        return x
    # convert to uppercase
    x = x.upper()
    # remove common prefixes and suffixes
    x = re.sub(r"\(TOWN OF\)|\(CITY OF\)|\(VILLAGE OF\)", "", x)
    x = re.sub(r"TOWN OF|CITY OF|VILLAGE OF", "", x)
    # remove any remaining parentheses
    x = re.sub(r"\(", "", x)
    x = re.sub(r"\)", "", x)
    # remove punctuation, extra whitespace, leading/trailing whitespace
    x = re.sub(r",", "", x)
    x = re.sub(r"\s+", " ", x)
    x = x.strip()
    # remove ' VT' or ' VERMONT' at the end, unless preceded by ' OF '
    # cleans some towns w/o affecting "Univ of Vermont", "State of Vermont", etc.
    # (?<! OF) means "not immediately preceded by ' OF'"
    x = re.sub(r"(?<! OF) (VT|VERMONT)$", "", x)
    return x


# apply cleaning function to subrecipient column to create town_clean
df["town_clean"] = df["subrecipient"].apply(clean_town)

# get and prepare canonical town list from ACS
town_list = acs_df["town_name"].str.upper().str.strip().unique().tolist()


# function for fuzzy matching
# uses rapidfuzz to find the best match in the town list, with a score cutoff to avoid bad matches
def match_town(name, town_list, score_cutoff=90):
    if pd.isna(name):
        return None
    match = process.extractOne(name, town_list, score_cutoff=score_cutoff)
    if match:
        return match[0]
    return None


# apply fuzzy matching to create town_match column
df["town_match"] = df["town_clean"].apply(lambda x: match_town(x, town_list))


# handle ambiguous cases, otherwise use fuzzy match
def resolve_town(row):
    # check for explicit classifications of ambiguous cases
    special = classify_barre_rutland_stalbans(row["subrecipient"])
    if special is not None:
        return special
    # then check manual overrides
    cleaned = row["town_clean"]
    if cleaned in manual_town_dict:
        return manual_town_dict[cleaned]
    # then check fuzzy match results
    elif pd.notna(row["town_match"]):
        return row["town_match"]
    else:
        return None


# create final town column using ambiguous case resolution and fuzzy matching
df["town_final"] = df.apply(resolve_town, axis=1)

# review ambiguous and unmatched cases
df["town_final"].value_counts(dropna=False)

town_final
None                    275
PAWLET TOWN              16
EAST MONTPELIER TOWN     13
RUPERT TOWN              11
NORTHFIELD TOWN          11
                       ... 
MONTPELIER CITY           1
BALTIMORE TOWN            1
STAMFORD TOWN             1
WOODSTOCK TOWN            1
GUILDHALL TOWN            1
Name: count, Length: 154, dtype: int64

Re: unmatched

Sherburne is either Shelburne or Killington  
Orleans can be a village in Barton or the county  
Chittenden, Lamoille, Windham, Windsor, Ottauquechee, could be distributed across the respective county / multiple towns  
Statewide stuff? Might just drop it. Could distribute per-capita.

In [17]:
# review ambiguous and unmatched cases
ambig = df[df["town_final"].str.startswith("AMBIGUOUS", na=False)]
print("Ambiguous subrecipients:", ambig[["subrecipient", "town_clean"]])

pd.set_option("display.max_rows", None)
# display where town_final is null to check unmatched cases
unmatched = df[df["town_final"].isna()]
print(
    f'\n\nNumber of unmatched cases: {len(unmatched[["subrecipient", "town_clean", "town_match", "town_final"]].drop_duplicates().sort_values("town_clean"))}'
)
unmatched[
    ["subrecipient", "town_clean", "town_match", "town_final"]
].drop_duplicates().sort_values("town_clean")

Ambiguous subrecipients:     subrecipient  town_clean
8          Barre       BARRE
19         Barre       BARRE
36       Rutland     RUTLAND
41         Barre       BARRE
52         Barre       BARRE
63         Barre       BARRE
119      Rutland     RUTLAND
170      Rutland     RUTLAND
172        Barre       BARRE
174        Barre       BARRE
176      Rutland     RUTLAND
179      Rutland     RUTLAND
183        Barre       BARRE
609        Barre       BARRE
696   St. Albans  ST. ALBANS
741   St. Albans  ST. ALBANS


Number of unmatched cases: 42


,subrecipient,town_clean,town_match,town_final
11,Chittenden (County),CHITTENDEN COUNTY,None,None
376,Chittenden County Regional Planning Commission,CHITTENDEN COUNTY REGIONAL PLANNING COMMISSION,None,None
424,CHITTENDEN REGIONAL PLANNING,CHITTENDEN REGIONAL PLANNING,None,None
413,"Department of Public Safety, Division of Emerg...",DEPARTMENT OF PUBLIC SAFETY DIVISION OF EMERGE...,None,None
471,DEPARTMENT OF PUBLIC SAFTEY,DEPARTMENT OF PUBLIC SAFTEY,None,None
248,DEPT. OF PUBLIC SAFE,DEPT. OF PUBLIC SAFE,None,None
406,ENVIRONMENTAL CONSERVATION,ENVIRONMENTAL CONSERVATION,None,None
309,Ethan Allen Homestead Trust,ETHAN ALLEN HOMESTEAD TRUST,None,None
208,Lamoille (County),LAMOILLE COUNTY,None,None
523,Northwest Regional Planning Commission,NORTHWEST REGIONAL PLANNING COMMISSION,None,None


In [18]:
pd.reset_option("display.max_rows")